In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from py2neo import Graph, Node, Relationship

In [ ]:
load_dotenv()

NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
DATA_DIR = os.getenv("DEMO_DATA_DIR")
RESULTS_DIR = os.getenv("RESULTS_DIR")

In [ ]:
IDS = ['subject_id', 'hadm_id', "icd_code", "icd_version", "itemid"]

# Load data from csv

In [ ]:
patients = pd.read_csv(os.path.join(DATA_DIR,"patients.csv.gz"))
admissions = pd.read_csv(os.path.join(DATA_DIR,"admissions.csv.gz"))
diagnoses = pd.read_csv(os.path.join(DATA_DIR,"diagnoses_icd.csv.gz"))
d_icd = pd.read_csv(os.path.join(DATA_DIR,"d_icd_diagnoses.csv.gz"))
labevents = pd.read_csv(os.path.join(DATA_DIR,"labevents.csv.gz"))
microbiologyevents = pd.read_csv(os.path.join(DATA_DIR,"microbiologyevents.csv.gz"))
d_labitems = pd.read_csv(os.path.join(DATA_DIR,"d_labitems.csv.gz"))

# Take look at data

In [ ]:
diagnoses.head()

In [ ]:
d_icd.head()

In [ ]:
patients.head()

In [ ]:
diagnoses.groupby('subject_id').size().describe()

In [ ]:
patients.shape, diagnoses.shape, admissions.shape

# Data selection

In [ ]:

patients_small = patients[["subject_id", "anchor_age", "gender", "anchor_year"]].drop_duplicates()
admissions_small = admissions[[
    "hadm_id","subject_id","admittime", "deathtime", "admission_type", 'admission_location', 
    'discharge_location', 'insurance', 'language', 'marital_status', 'race']].drop_duplicates()
diagnoses_small = diagnoses[["hadm_id","subject_id", "icd_code", "icd_version"]].drop_duplicates()

# patients_small = patients_small.head(1)
admissions_small = admissions_small[admissions_small['subject_id'].isin(patients_small['subject_id'])]
diagnoses_small = diagnoses_small[diagnoses_small['subject_id'].isin(patients_small['subject_id'])]

d_icd_small = d_icd.merge(
    diagnoses_small[['icd_code', 'icd_version']].drop_duplicates(),
    on=['icd_code', 'icd_version'],
    how='inner'
)

labevents_small = labevents.merge(
    admissions_small[['subject_id', 'hadm_id']].drop_duplicates(),
    on=['subject_id', 'hadm_id'],
    how='inner'
)
d_labitems_small = d_labitems.merge(
    labevents_small[['itemid']].drop_duplicates(),
    on=['itemid'],
    how='inner'
)

# Graph generation in Neo4j

In [ ]:
graph = Graph(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD), name='test2')
graph.delete_all()

## Nodes

In [ ]:
tx = graph.begin()

patient_nodes = {}
for _, r in patients_small.iterrows():
    attrs = {k: v for k, v in dict(r).items() if k not in IDS}
    # attrs = r.to_dict()
    n = Node("Patient", name=r.subject_id, subject_id=r.subject_id, **attrs)
    tx.create(n)
    patient_nodes[r.subject_id] = n

admission_nodes = {}
for _, r in admissions_small.iterrows():
    attrs = {k: v for k, v in dict(r).items() if k not in IDS}
    n = Node("Admission", name=r.hadm_id, hadm_id = r.hadm_id, **attrs)
    tx.create(n)
    admission_nodes[r.hadm_id] = n

d_icd_nodes = {}
for _, r in d_icd_small.iterrows():
    attrs = {k: v for k, v in dict(r).items() if k not in IDS}
    n = Node(
        "Diagnosis", 
        name=r.long_title, icd_version = r.icd_version, icd_code = r.icd_code, 
        **attrs)
    tx.create(n)
    d_icd_nodes[(r.icd_code, r.icd_version)] = n

d_labitems_nodes = {}
for _, r in d_labitems_small.iterrows():
    attrs = {k: v for k, v in dict(r).items() if k not in IDS}
    n = Node("LabItem", name=r.label, itemid=r.itemid, **attrs)
    tx.create(n)
    d_labitems_nodes[r.itemid] = n

graph.commit(tx)

## Edges

In [ ]:
tx = graph.begin()

for _, r in admissions_small.iterrows():
    subject_id = r.subject_id
    hadm_id = r.hadm_id
    patient_node = node = graph.nodes.match("Patient", subject_id=subject_id).first()
    admission_node = graph.nodes.match("Admission", hadm_id=hadm_id).first()
    if patient_node and admission_node:
        rel = Relationship(patient_node, "HAS_ADMISSION", admission_node)
        graph.create(rel)

for _, r in diagnoses_small.iterrows():
    admission_node = graph.nodes.match(
            "Admission", 
            hadm_id=r.hadm_id
        ).first()
    diagnosis_node = graph.nodes.match(
            "Diagnosis", 
            icd_code=r.icd_code, icd_version=r.icd_version
        ).first()

    if admission_node and diagnosis_node:
        rel = Relationship(admission_node, "HAS_DIAGNOSIS", diagnosis_node)
        graph.create(rel)

for _, r in labevents_small.iterrows():

    exists = graph.evaluate(
        """
        MATCH (a:Admission {hadm_id: $hadm_id})-[r:HAS_LABEVENT]->(b:LabItem {itemid: $itemid})
        RETURN COUNT(r) > 0
        """,
        hadm_id=r.hadm_id,
        itemid=r.itemid
    )
    if not exists:
        
        admission_node = graph.nodes.match(
                "Admission", 
                hadm_id=r.hadm_id
            ).first()
        lebevent_node = graph.nodes.match(
                "LabItem", 
                itemid=r.itemid
            ).first()

        if admission_node and lebevent_node:
            attrs = {k: v for k, v in dict(r).items() if k not in IDS}
            rel = Relationship(
                admission_node, "HAS_LABEVENT", lebevent_node,
                **attrs)
            graph.create(rel)

graph.commit(tx)

In [ ]:
# Alternative approach with cached nodes

# for _, r in admissions_small.iterrows():
#     subject_id = r.subject_id
#     hadm_id = r.hadm_id
#     patient_node = patient_nodes.get(subject_id)
#     admission_node = admission_nodes.get(hadm_id)
#     if patient_node and admission_node:
#         rel = Relationship(patient_node, "HAS_ADMISSION", admission_node)
#         graph.create(rel)

# for _, r in diagnoses_small.iterrows():
#     hadm_id = r.hadm_id
#     icd_code = r.icd_code
#     icd_version = r.icd_version
#     admission_node = admission_nodes.get(hadm_id)
#     d_node = d_icd_nodes.get((icd_code, icd_version))
#     if admission_node and d_node:
#         rel = Relationship(admission_node, "HAS_DIAGNOSIS", d_node)
#         graph.create(rel)

# graph.commit(tx)